In [48]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator
from langchain_core.messages import SystemMessage, HumanMessage


In [49]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=os.getenv("GOOGLE_GENAI_KEY"),
)

In [50]:
class EvaluationScehma(BaseModel):
    evaluation : Literal['approved', 'need_approval'] = Field(description="Final Evaluation of the tweet")
    feedback: str = Field(description="Feedback of the tweet generated")

In [51]:
strctured_llm = llm.with_structured_output(EvaluationScehma)

In [56]:
class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation : Literal['approved', 'needs_improvement']
    feedback: str
    iteration: int
    max_iteration: int

    tweet_history : Annotated[list[str], operator.add]
    feedback_history : Annotated[list[str], operator.add]

In [57]:
def generate_tweet(state: TweetState) -> dict:

    messages = f'Generate a tweet in {state['topic']}'

    tweet = llm.invoke(messages).content

    return {
        'tweet': tweet,
        'tweet_history': [tweet]
    }
def evaluate_tweet(state: TweetState) -> dict:
    messages = [
            SystemMessage(
                content="""
        You are an expert Twitter (X) content reviewer.

        Evaluate the tweet based on:
        - Humor and originality
        - Clarity and readability
        - Engagement potential
        - Relevance to the given topic
        - Whether it is appropriate for a general audience

        Decision criteria:
        - Return "approved" if the tweet is funny, engaging, relevant, and ready to post.
        - Return "needs_revision" if the tweet is unfunny, unclear, generic, off-topic, or could be significantly improved.

        Provide concise and actionable feedback. If approved, briefly explain why. If it needs revision, explain what should be improved.

        Return the response according to the provided structured output schema.
        """
            ),
            HumanMessage(
                content=f"""
        Topic:
        {state["topic"]}

        Tweet:
        {state["tweet"]}

        Evaluate this tweet.
        """
            ),
        ]

    response = strctured_llm.invoke(messages)

    return {
        'evaluation': response.evaluation,
        'feedback': response.feedback,
        'feedback_history': [response]
    }
def optimize_tweet(state: TweetState) -> dict:
    messages = [
    SystemMessage(
        content="""
You are an expert Twitter (X) copywriter.

Your task is to rewrite tweets to maximize humor, originality, and engagement while preserving the original intent and topic.

Guidelines:
- Carefully incorporate all provided feedback.
- Keep the core idea of the original tweet unless the feedback indicates it should change.
- Make the tweet funnier, more natural, and more engaging.
- Keep it under 280 characters.
- Avoid repetitive wording, clichés, and forced humor.
- Do not include explanations, notes, or quotation marks.
- Return only the improved tweet.
"""
    ),
    HumanMessage(
        content=f"""
Topic:
{state["topic"]}

Original Tweet:
{state["tweet"]}

Feedback:
{state["feedback"]}

Rewrite the tweet by addressing every point in the feedback while keeping the tweet relevant to the topic and improving its quality.
"""
    ),
]

    response = llm.invoke(messages).content
    iteration = state['iteration']+1
    return {
        'tweet': response,
        'iteration': iteration,
        'tweet_history': [response]
    }

def checkEvaluation(state: TweetState):
    evaluation = state['evaluation']
    iteration = state['iteration']
    max_iteration = state['max_iteration']

    if evaluation == 'approved' or iteration >= max_iteration:
        return 'approved'
    else:
        return 'needs_improvement'

In [58]:
graph = StateGraph(TweetState)

#Add Node
graph.add_node('generate', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize', optimize_tweet)

#Add Edges
graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')
graph.add_conditional_edges('evaluate', checkEvaluation, {'approved': END , 'needs_improvement': 'optimize'})
graph.add_edge('optimize', 'evaluate')

worfklow = graph.compile()

In [59]:
initial_state = {
    'topic': 'AI',
    'iteration': 1,
    'max_iteration': 3
}

final_state = worfklow.invoke(initial_state)

final_state

{'topic': 'AI',
 'tweet': [{'type': 'text',
   'text': 'I told myself I’d use AI to automate my life and reach peak productivity. \n\nInstead, I’ve spent the last 4 hours debating with a chatbot about whether a cat eating pizza in space is anatomically correct. 🐈🍕✨ #AI #PromptEngineering',
   'extras': {'signature': 'EjQKMgERTTIPWH5zVWwWdSWFGLtjZgPxVvqeL72rALxgiKE1hFWiUou0T8HK6ePPr7hEqnIc'}}],
 'evaluation': 'approved',
 'feedback': 'This tweet is highly relatable, funny, and captures the common reality of AI experimentation. It uses appropriate hashtags and effectively engages the reader.',
 'iteration': 2,
 'max_iteration': 3,
 'tweet_history': [[{'type': 'text',
    'text': 'Here are a few options for a tweet about AI, depending on the vibe you want:\n\n**The Optimistic/Visionary approach:**\n"AI isn’t here to replace human creativity; it’s here to act as a force multiplier for our imagination. We’re moving from the era of \'how to build\' to the era of \'what to dream.\' The future